# 05 — Thresholds de risco em três níveis (Fase 6)

Cada transação recebe um score e cai em um de três níveis:
**aprovar automaticamente**, **pedir autenticação adicional** ou **encaminhar para investigação**.

- Usa os **scores do XGBoost salvos pelo notebook 04** (`reports/scores`). Nada é retreinado: roda em segundos, **CPU basta**.
- Os dois cortes são escolhidos na **validação**, minimizando o custo esperado (custo operacional + perda das fraudes não barradas),
  e medidos no **teste**. O ótimo é exato (busca sobre todos os cortes possíveis).
- Premissas do `config.yaml`: investigação R$ 50 (captura 95%), autenticação R$ 2 (captura 70%). São premissas, não dados reais.
  Como o valor médio de fraude do PaySim é muito maior que esses custos, rodamos também uma **grade de custos** (mesma razão 25:1).
- Também medimos o **"arrependimento"**: quanto se perde no teste por ter escolhido o corte na validação, em vez do corte ideal do teste.
  (No notebook 04, o corte do Random Forest não se transferiu bem entre validação e teste.)

In [ ]:
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures'); os.makedirs(FIG, exist_ok=True)
MODELS = cfg['paths']['models_dir']; os.makedirs(MODELS, exist_ok=True)
SCORES = os.path.join(cfg['paths']['reports_dir'], 'scores'); os.makedirs(SCORES, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

In [ ]:
MODELO = 'xgboost'
CENARIOS = ['B', 'C']
PARES_CUSTO = [(50, 2), (500, 20), (5000, 200), (50000, 2000)]   # (custo de investigação, custo de autenticação)
PADRAO = (cfg['costs']['investigation_cost'], cfg['costs']['authentication_cost'])
INV_CATCH = cfg['costs']['investigation_catch_rate']
AUTH_CATCH = cfg['costs']['authentication_catch_rate']
FINAL = os.path.join(MODELS, 'final'); os.makedirs(FINAL, exist_ok=True)
print('custos padrão (investigação, autenticação):', PADRAO, '| captura:', INV_CATCH, AUTH_CATCH)

## 1. Dados e scores salvos pelo notebook 04

In [ ]:
from fraud.data import load_parquet
from fraud.split import make_split
from fraud.costs import legit_weight_for
from fraud.thresholds import optimize_levels, evaluate_levels, decision_levels

df = load_parquet(cfg['paths']['parquet'])
_, val, test = make_split(df, cfg)
del df
yv, yt = val[y].values, test[y].values
av, at = val['amount'].values, test['amount'].values
wv, wt = legit_weight_for(yv), legit_weight_for(yt)
sv = {l: np.load(os.path.join(SCORES, f'val_{MODELO}_{l}_final.npy')).astype('float64') for l in CENARIOS}
st = {l: np.load(os.path.join(SCORES, f'test_{MODELO}_{l}_final.npy')).astype('float64') for l in CENARIOS}
print(f'validação: {len(yv):,} linhas, {int(yv.sum())} fraudes | teste: {len(yt):,} linhas, {int(yt.sum())} fraudes')
print(f'pesos de reponderação: validação {wv:.2f} | teste {wt:.2f}')

## 2. Cortes ótimos para todos os pares de custo
`t_auth` e `t_inv` são cortes no score (`inf` = o nível nunca é usado). As colunas de nível mostram a **% de transações reais**
em cada nível (falsos positivos reponderados para a taxa de fraude real de 0,129%).

In [ ]:
def rodar(letra, inv_cost, auth_cost):
    kw = dict(inv_cost=inv_cost, auth_cost=auth_cost, inv_catch=INV_CATCH, auth_catch=AUTH_CATCH)
    ot = optimize_levels(yv, av, sv[letra], legit_weight=wv, **kw)
    ev = evaluate_levels(yt, at, st[letra], ot['t_auth'], ot['t_inv'], legit_weight=wt, **kw)
    ev2 = evaluate_levels(yt, at, st[letra], ot['t_two'], ot['t_two'], legit_weight=wt, **kw)
    orc = optimize_levels(yt, at, st[letra], legit_weight=wt, **kw)
    return ot, ev, ev2, orc

linhas, detalhe = [], {}
for letra in CENARIOS:
    for ic, ac in PARES_CUSTO:
        ot, ev, ev2, orc = rodar(letra, ic, ac)
        detalhe[(letra, ic, ac)] = (ot, ev)
        n = {r['nivel']: r for r in ev['niveis']}
        linhas.append({
            'cenario': letra, 'custo_invest': ic, 'custo_auth': ac,
            't_auth': ot['t_auth'], 't_inv': ot['t_inv'],
            '%_autenticar': n['autenticar']['pct_transacoes_reais'],
            '%_investigar': n['investigar']['pct_transacoes_reais'],
            'fraudes_aprovadas': n['aprovar']['fraudes'],
            'fraudes_autenticadas': n['autenticar']['fraudes'],
            'fraudes_investigadas': n['investigar']['fraudes'],
            'valor_poupado_%': ev['valor_poupado_%'],
            'beneficio_teste': ev['beneficio_liquido'],
            'beneficio_2_niveis': ev2['beneficio_liquido'],
            'beneficio_ideal_teste': orc['beneficio_liquido'],
            'arrependimento_%': 100 * (orc['beneficio_liquido'] - ev['beneficio_liquido']) / orc['beneficio_liquido'],
        })
tab = pd.DataFrame(linhas).set_index(['cenario', 'custo_invest', 'custo_auth'])
tab.to_csv(os.path.join(cfg['paths']['reports_dir'], 'final_thresholds_3_niveis.csv'))
pd.options.display.float_format = '{:,.4f}'.format
for letra in CENARIOS:
    print(f'Cenário {letra}')
    display(tab.loc[letra][['t_auth', 't_inv', '%_autenticar', '%_investigar', 'fraudes_aprovadas',
                            'fraudes_autenticadas', 'fraudes_investigadas', 'valor_poupado_%']])

## 3. Quanto vale o terceiro nível e quanto se perde por escolher o corte na validação
`beneficio_teste` é o resultado dos cortes escolhidos na validação. `beneficio_2_niveis` usa só investigação.
`arrependimento_%` compara com o melhor corte possível no próprio teste.

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
for letra in CENARIOS:
    print(f'Cenário {letra}')
    t = tab.loc[letra][['beneficio_teste', 'beneficio_2_niveis', 'beneficio_ideal_teste', 'arrependimento_%']].copy()
    t['ganho_3_niveis_vs_2'] = t['beneficio_teste'] - t['beneficio_2_niveis']
    display(t)

## 4. Os três níveis no caso padrão (teste)

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
for letra in CENARIOS:
    ot, ev = detalhe[(letra, *PADRAO)]
    print(f'Cenário {letra} | custo padrão {PADRAO} | t_auth = {ot["t_auth"]:.6g} | t_inv = {ot["t_inv"]:.6g}')
    d = pd.DataFrame(ev['niveis']).set_index('nivel')
    d['precisao_%'] = 100 * d['fraudes'] / d['transacoes'].replace(0, np.nan)
    display(d[['transacoes', 'fraudes', 'falsos_positivos', 'precisao_%', 'valor_fraude', 'pct_transacoes_reais']])
    print(f"   valor poupado {ev['valor_poupado_%']:.2f}% | benefício líquido {ev['beneficio_liquido']:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
for letra, cor in zip(CENARIOS, ['tab:blue', 'tab:orange']):
    s, a = st[letra], at
    o = np.argsort(-s, kind='mergesort')
    frac = np.arange(1, len(o) + 1) / len(o)
    ganho = np.cumsum(a[o] * yt[o]) / a[yt == 1].sum()
    ax.plot(frac, ganho, color=cor, label=f'cenário {letra}')
    ot, _ = detalhe[(letra, *PADRAO)]
    for t, marca, nome in [(ot['t_auth'], '^', 'autenticar'), (ot['t_inv'], 'o', 'investigar')]:
        if np.isfinite(t):
            k = int((s >= t).sum())
            if k:
                ax.scatter(k / len(s), ganho[k - 1], color=cor, marker=marca, s=60, zorder=3,
                           label=f'{letra}: início de {nome}')
ax.set_xscale('log'); ax.set_xlabel('fração das transações sinalizadas (teste)')
ax.set_ylabel('fração do valor fraudado capturado'); ax.set_title('Curva de ganho no teste (custo padrão)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
t = tab.loc['C']
x = np.arange(len(t)); w = 0.38
ax.bar(x - w / 2, t['%_autenticar'], w, label='autenticar')
ax.bar(x + w / 2, t['%_investigar'], w, label='investigar')
ax.set_yscale('log'); ax.set_xticks(x)
ax.set_xticklabels([f'{i:,}/{a:,}' for i, a in zip(t.index.get_level_values(0), t.index.get_level_values(1))])
ax.set_xlabel('custo de investigação / autenticação'); ax.set_ylabel('% das transações reais')
ax.set_title('Cenário C: carga operacional por nível'); ax.legend(); ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'final_thresholds.png'), dpi=130); plt.show()

## 5. Salvar os cortes para a API e o painel

In [ ]:
import json
def num(v):
    return None if not np.isfinite(v) else float(v)

saida = {'modelo': MODELO,
         'premissas': {'investigation_catch_rate': INV_CATCH, 'authentication_catch_rate': AUTH_CATCH,
                       'taxa_fraude_real': 0.001291},
         'observacao': 'null = nível nunca usado. Cortes escolhidos na validação; scores do XGBoost do notebook 04.',
         'thresholds': {}}
for letra in CENARIOS:
    saida['thresholds'][letra] = {}
    for ic, ac in PARES_CUSTO:
        ot, _ = detalhe[(letra, ic, ac)]
        saida['thresholds'][letra][f'inv{ic}_auth{ac}'] = {
            't_auth': num(ot['t_auth']), 't_inv': num(ot['t_inv']),
            'custo_investigacao': ic, 'custo_autenticacao': ac, 'padrao': (ic, ac) == tuple(PADRAO)}
arq = os.path.join(FINAL, 'thresholds.json')
json.dump(saida, open(arq, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('salvo em', arq)

## 6. Exportar todas as saídas deste notebook em um .txt

In [ ]:
try:
    from google.colab import _message, files
    nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)['ipynb']
    partes = []
    for c in nb['cells']:
        if c['cell_type'] != 'code':
            continue
        for o in c.get('outputs', []):
            if o.get('output_type') == 'stream':
                partes.append(''.join(o['text']))
            elif 'data' in o and 'text/plain' in o['data']:
                partes.append(''.join(o['data']['text/plain']))
    txt = os.path.join(cfg['paths']['reports_dir'], 'saidas_notebook_05.txt')
    open(txt, 'w', encoding='utf-8').write('\n'.join(partes))
    files.download(txt)
except ImportError:
    print('Fora do Colab: exportação ignorada.')

## O que enviar de volta
O arquivo `saidas_notebook_05.txt` (seção 6) e a imagem `final_thresholds.png` (em `reports/figures`).